# 10 - Operational Prioritization

## Objective

This notebook transforms model predictions into operational recommendations by scoring the September–October validation window, ranking high-risk flights, applying capacity-constrained prioritization logic, and evaluating prioritized selection against a random baseline for RQ4/H4.

Outputs support the Streamlit prioritization tab, the final report, and downstream dashboard preparation.

#### Load project configuration

In [0]:
from config import project_config as cfg

print("Project configuration loaded successfully.")
print(f"Predictions table: {cfg.PREDICTIONS_TABLE}")
print(f"Prioritization table: {cfg.PRIORITIZATION_RESULTS_TABLE}")
print(f"Scoring window: {cfg.SCORING_START_DATE} to {cfg.SCORING_END_DATE}")


Project configuration loaded successfully.
Predictions table: workspace.default.flight_predictions
Prioritization table: workspace.default.flight_prioritization_results
Scoring window: 2025-09-01 to 2025-10-31


#### Load the saved model and modeling checkpoints

The selected Spark ML model and the validation-period modeling datasets produced in notebooks 07 and 08 are loaded before batch scoring.

In [0]:
from __future__ import annotations

import json

from pyspark.ml.classification import LogisticRegressionModel
from pyspark.sql import functions as F
from utils.model_training import (
    create_feature_hasher_from_manifest,
    hash_modeling_frame,
    load_hist_modeling_table,
)


def require_table(table_name: str) -> None:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Required table '{table_name}' was not found. "
            "Run notebooks 07, 08, and 09 before continuing."
        )


def require_file(file_path: str) -> None:
    try:
        dbutils.fs.head(file_path, 1)
    except Exception as exc:
        raise RuntimeError(
            f"Required file '{file_path}' was not found."
        ) from exc

def require_model_path(model_path: str) -> None:
    try:
        dbutils.fs.ls(model_path)
    except Exception as exc:
        raise RuntimeError(
            f"Required model path '{model_path}' was not found. "
            "Run notebook 08 through the model-save cells."
        ) from exc

require_model_path(cfg.SELECTED_MODEL_PATH)
require_file(cfg.SELECTED_MODEL_METRICS_PATH)
require_file(cfg.MODEL_FEATURE_MANIFEST_PATH)

for table_name in [
    cfg.MODELING_VALIDATION_HIST_TABLE,
    cfg.SHAP_GLOBAL_IMPORTANCE_TABLE,
]:
    require_table(table_name)

feature_manifest = json.loads(
    dbutils.fs.head(cfg.MODEL_FEATURE_MANIFEST_PATH, 1000000)
)
model_metrics = json.loads(
    dbutils.fs.head(cfg.SELECTED_MODEL_METRICS_PATH, 1000000)
)

TARGET_COLUMN = feature_manifest["target_column"]
DECISION_THRESHOLD = float(
    model_metrics.get(
        "selected_validation_threshold",
        cfg.DEFAULT_DECISION_THRESHOLD,
    )
)

final_model = LogisticRegressionModel.load(cfg.SELECTED_MODEL_PATH)
df_validation_hist = load_hist_modeling_table(cfg.MODELING_VALIDATION_HIST_TABLE)
feature_hasher = create_feature_hasher_from_manifest(feature_manifest)
df_validation_hashed = hash_modeling_frame(
    df_validation_hist,
    feature_hasher,
)
top_shap_feature = (
    spark.table(cfg.SHAP_GLOBAL_IMPORTANCE_TABLE)
    .orderBy(F.col("MeanAbsSHAP").desc())
    .limit(1)
    .collect()[0]["Feature"]
)

print("Saved model and checkpoints loaded successfully.")
print(f"Decision threshold: {DECISION_THRESHOLD:.2f}")
print(f"Model input columns: {len(feature_manifest['model_input_columns'])}")
print(f"Top SHAP driver: {top_shap_feature}")


[Truncated to first 1 bytes]
[Truncated to first 1 bytes]
Saved model and checkpoints loaded successfully.
Decision threshold: 0.20
Model input columns: 24
Top SHAP driver: FLIGHT DISTANCE CATEGORY Medium


#### Score the operational review window

The September–October validation window is used as the operational scoring period because it contains completed flights with known delay outcomes and supports RQ4 evaluation without using the final holdout test set.

In [0]:
from pyspark.ml.functions import vector_to_array

scored_predictions = final_model.transform(df_validation_hashed)

scored_predictions = scored_predictions.select(
    *cfg.MODELING_JOIN_KEY_COLUMNS,
    F.col(TARGET_COLUMN).alias("actual_delay"),
    F.col("prediction").alias("predicted_delay"),
    F.element_at(
        vector_to_array(F.col("probability")),
        2,
    ).alias("delay_probability"),
)

display(scored_predictions.limit(5))
print(f"Scored flights: {scored_predictions.count():,}")


FL_DATE,OP_UNIQUE_CARRIER,ORIGIN,DEST,CRS_DEP_TIME,actual_delay,predicted_delay,delay_probability
2025-09-01,AA,JFK,LAX,700,0,0.0,0.22445439605159223
2025-09-01,AA,LAX,JFK,2121,0,0.0,0.38788719741965827
2025-09-01,AA,MSN,CLT,716,0,0.0,0.2172479642522821
2025-09-01,AA,CLT,MCI,1615,1,0.0,0.4846818282200609
2025-09-01,AA,MCI,CLT,1833,1,1.0,0.5138177834215263


Scored flights: 1,159,898


In [0]:
import pandas as pd

from utils.operational_prioritization import add_operational_scores


prediction_attributes = (
    df_validation_hist.select(
        *cfg.MODELING_JOIN_KEY_COLUMNS,
        cfg.AIRLINE_COLUMN,
        cfg.ORIGIN_COLUMN,
        cfg.DESTINATION_COLUMN,
        cfg.SCHEDULED_DEPARTURE_COLUMN,
        cfg.MONTH_COLUMN,
        cfg.TIME_OF_DAY_COLUMN,
        cfg.SEASON_COLUMN,
    )
)

predictions_frame = (
    scored_predictions.join(
        prediction_attributes,
        on=cfg.MODELING_JOIN_KEY_COLUMNS,
        how="inner",
    )
    .withColumn(
        "flight_label",
        F.concat_ws(
            "-",
            F.col(cfg.AIRLINE_COLUMN),
            F.col(cfg.ORIGIN_COLUMN),
            F.col(cfg.DESTINATION_COLUMN),
            F.col(cfg.SCHEDULED_DEPARTURE_COLUMN).cast("string"),
        ),
    )
    .withColumn(
        "scheduled_departure_text",
        F.date_format(
            F.to_timestamp(
                F.lpad(F.col(cfg.SCHEDULED_DEPARTURE_COLUMN).cast("string"), 4, "0"),
                "HHmm",
            ),
            "HH:mm",
        ),
    )
    .withColumn("shap_main_driver", F.lit(top_shap_feature))
)

predictions_pdf = predictions_frame.toPandas()
predictions_pdf = predictions_pdf.rename(
    columns={
        cfg.AIRLINE_COLUMN: "airline_code",
        cfg.ORIGIN_COLUMN: "origin_airport",
        cfg.DESTINATION_COLUMN: "destination_airport",
        cfg.SCHEDULED_DEPARTURE_COLUMN: "scheduled_departure",
        cfg.MONTH_COLUMN: "month_number",
        cfg.TIME_OF_DAY_COLUMN: "departure_window",
        cfg.SEASON_COLUMN: "season",
    }
)
predictions_pdf = add_operational_scores(
    predictions_pdf,
    high_threshold=cfg.HIGH_RISK_THRESHOLD,
    critical_threshold=cfg.CRITICAL_RISK_THRESHOLD,
    medium_threshold=cfg.MEDIUM_RISK_THRESHOLD,
)

display(spark.createDataFrame(predictions_pdf).limit(5))
print(f"Operational predictions prepared: {len(predictions_pdf):,}")


FL_DATE,airline_code,origin_airport,destination_airport,scheduled_departure,actual_delay,predicted_delay,delay_probability,month_number,departure_window,season,flight_label,scheduled_departure_text,shap_main_driver,risk_level,priority_score,recommendation
2025-09-01,AA,ALB,CLT,1136,0,0.0,0.3121232518562844,9,Morning,Fall,AA-ALB-CLT-1136,11:36,FLIGHT DISTANCE CATEGORY Medium,MEDIUM,31,Increased Operational Monitoring
2025-09-01,AA,ALB,CLT,1344,0,0.0,0.3902488210457933,9,Afternoon,Fall,AA-ALB-CLT-1344,13:44,FLIGHT DISTANCE CATEGORY Medium,MEDIUM,39,Increased Operational Monitoring
2025-09-01,AA,ANC,DFW,1946,1,1.0,0.552314577918042,9,Evening,Fall,AA-ANC-DFW-1946,19:46,FLIGHT DISTANCE CATEGORY Medium,MEDIUM,55,Increased Operational Monitoring
2025-09-01,AA,ATL,DFW,1110,0,0.0,0.46307225993919465,9,Morning,Fall,AA-ATL-DFW-1110,11:10,FLIGHT DISTANCE CATEGORY Medium,MEDIUM,46,Increased Operational Monitoring
2025-09-01,AA,ATL,PHL,1528,0,1.0,0.5775364944562986,9,Afternoon,Fall,AA-ATL-PHL-1528,15:28,FLIGHT DISTANCE CATEGORY Medium,MEDIUM,58,Increased Operational Monitoring


Operational predictions prepared: 1,160,866


In [0]:
from utils.operational_prioritization import (
    build_ranking_table,
    compare_prioritization_strategies,
)


prioritization_pool = predictions_pdf[
    predictions_pdf["delay_probability"] >= cfg.PRIORITIZATION_POOL_MIN_PROB
].copy()

ranking_tables = []
evaluation_tables = []

for capacity_k in cfg.CAPACITY_K_OPTIONS:
    ranking = build_ranking_table(
        prioritization_pool,
        capacity_k=capacity_k,
        airline_column="airline_code",
        origin_column="origin_airport",
    )
    ranking["capacity_k"] = capacity_k
    ranking_tables.append(ranking)

    evaluation = compare_prioritization_strategies(
        prioritization_pool,
        capacity_k=capacity_k,
        random_seed=cfg.RANDOM_SEED,
        label_column="actual_delay",
        airline_column="airline_code",
        origin_column="origin_airport",
    )
    evaluation_tables.append(evaluation)

prioritization_results_pdf = pd.concat(ranking_tables, ignore_index=True)
prioritization_evaluation_pdf = pd.concat(evaluation_tables, ignore_index=True)

display(
    spark.createDataFrame(
        prioritization_evaluation_pdf[
            prioritization_evaluation_pdf["capacity_k"] == cfg.DEFAULT_CAPACITY_K
        ]
    )
)
display(
    spark.createDataFrame(
        prioritization_results_pdf[
            (prioritization_results_pdf["capacity_k"] == cfg.DEFAULT_CAPACITY_K)
            & (prioritization_results_pdf["selected"])
        ].head(20)
    )
)


capacity_k,strategy,population_size,selected_count,total_delayed_flights,captured_delayed_flights,delay_recall,delay_precision,lift_vs_random
25,Prioritized Selection,78602.0,25.0,27763.0,13.0,4.682491085257357E-4,0.52,1.472212657133595
25,Random Baseline,78602.0,25.0,27763.0,9.0,3.241724597485863E-4,0.36,1.019224147246335


FL_DATE,airline_code,origin_airport,destination_airport,scheduled_departure,actual_delay,predicted_delay,delay_probability,month_number,departure_window,season,flight_label,scheduled_departure_text,shap_main_driver,risk_level,priority_score,recommendation,selected,priority_rank,capacity_k
2025-10-31,G4,ROA,SFB,2043,1,1.0,0.8434856448594626,10,Evening,Fall,G4-ROA-SFB-2043,20:43,FLIGHT DISTANCE CATEGORY Medium,CRITICAL,84,Immediate Operational Assessment,true,1,25
2025-10-03,G4,SFB,USA,1946,0,1.0,0.8357206637642616,10,Evening,Fall,G4-SFB-USA-1946,19:46,FLIGHT DISTANCE CATEGORY Medium,CRITICAL,84,Immediate Operational Assessment,true,2,25
2025-10-17,G4,SFB,USA,1946,0,1.0,0.8357206637642616,10,Evening,Fall,G4-SFB-USA-1946,19:46,FLIGHT DISTANCE CATEGORY Medium,CRITICAL,84,Immediate Operational Assessment,true,3,25
2025-10-10,G4,SFB,USA,1946,0,1.0,0.8357206637642616,10,Evening,Fall,G4-SFB-USA-1946,19:46,FLIGHT DISTANCE CATEGORY Medium,CRITICAL,84,Immediate Operational Assessment,true,4,25
2025-10-05,AA,ORD,EWR,2041,0,1.0,0.824783204026716,10,Evening,Fall,AA-ORD-EWR-2041,20:41,FLIGHT DISTANCE CATEGORY Medium,CRITICAL,82,Immediate Operational Assessment,true,28,25
2025-10-19,AA,ORD,EWR,2035,0,1.0,0.8228813601177422,10,Evening,Fall,AA-ORD-EWR-2035,20:35,FLIGHT DISTANCE CATEGORY Medium,CRITICAL,82,Immediate Operational Assessment,true,29,25
2025-10-12,AA,ORD,EWR,2035,1,1.0,0.8228813601177422,10,Evening,Fall,AA-ORD-EWR-2035,20:35,FLIGHT DISTANCE CATEGORY Medium,CRITICAL,82,Immediate Operational Assessment,true,30,25
2025-10-26,AA,ORD,EWR,2035,1,1.0,0.8228813601177422,10,Evening,Fall,AA-ORD-EWR-2035,20:35,FLIGHT DISTANCE CATEGORY Medium,CRITICAL,82,Immediate Operational Assessment,true,31,25
2025-10-19,OH,MGM,DCA,1808,0,1.0,0.8149010945124429,10,Evening,Fall,OH-MGM-DCA-1808,18:08,FLIGHT DISTANCE CATEGORY Medium,CRITICAL,81,Immediate Operational Assessment,true,53,25
2025-10-12,OH,MGM,DCA,1808,1,1.0,0.8149010945124429,10,Evening,Fall,OH-MGM-DCA-1808,18:08,FLIGHT DISTANCE CATEGORY Medium,CRITICAL,81,Immediate Operational Assessment,true,54,25


#### Validate RQ4 / H4

RQ4 is supported when prioritized selection captures more delayed flights than a random baseline at the same operational capacity K. The comparison below provides the statistical evidence for the final report.

In [0]:
rq4_summary = prioritization_evaluation_pdf.copy()
rq4_summary["rq4_supported"] = (
    rq4_summary.groupby("capacity_k")["captured_delayed_flights"].transform("max")
    == rq4_summary["captured_delayed_flights"]
) & (rq4_summary["strategy"] == "Prioritized Selection")

display(spark.createDataFrame(rq4_summary))

default_k_results = rq4_summary[
    rq4_summary["capacity_k"] == cfg.DEFAULT_CAPACITY_K
]
prioritized_row = default_k_results[
    default_k_results["strategy"] == "Prioritized Selection"
].iloc[0]
random_row = default_k_results[
    default_k_results["strategy"] == "Random Baseline"
].iloc[0]

print(
    "RQ4 default-K comparison: "
    f"prioritized captured {int(prioritized_row['captured_delayed_flights'])} delays vs "
    f"random {int(random_row['captured_delayed_flights'])} delays "
    f"at K={cfg.DEFAULT_CAPACITY_K}."
)


capacity_k,strategy,population_size,selected_count,total_delayed_flights,captured_delayed_flights,delay_recall,delay_precision,lift_vs_random,rq4_supported
10,Prioritized Selection,78602.0,10.0,27763.0,4.0,1.4407664877714944E-4,0.4,1.13247127471815,true
10,Random Baseline,78602.0,10.0,27763.0,2.0,7.203832438857472E-5,0.2,0.566235637359075,false
25,Prioritized Selection,78602.0,25.0,27763.0,13.0,4.682491085257357E-4,0.52,1.472212657133595,true
25,Random Baseline,78602.0,25.0,27763.0,9.0,3.241724597485863E-4,0.36,1.019224147246335,false
50,Prioritized Selection,78602.0,50.0,27763.0,24.0,8.644598926628967E-4,0.48,1.35896552966178,true
50,Random Baseline,78602.0,50.0,27763.0,19.0,6.843640816914598E-4,0.38,1.0758477109822424,false
100,Prioritized Selection,78602.0,56.0,27763.0,26.0,9.364982170514714E-4,0.4642857142857143,1.3144755867264242,false
100,Random Baseline,78602.0,100.0,27763.0,35.0,0.0012606706768000576,0.35,0.9909123653783812,false


RQ4 default-K comparison: prioritized captured 13 delays vs random 9 delays at K=25.


#### Save operational outputs

In [0]:
predictions_df = spark.createDataFrame(predictions_pdf)
prioritization_results_df = spark.createDataFrame(prioritization_results_pdf)
prioritization_evaluation_df = spark.createDataFrame(prioritization_evaluation_pdf)

(
    predictions_df.write.format("delta").mode("overwrite").save(cfg.PREDICTIONS_DELTA_PATH)
)
(
    predictions_df.writeTo(cfg.PREDICTIONS_TABLE).using("delta").createOrReplace()
)

(
    prioritization_results_df.write.format("delta").mode("overwrite").save(
        cfg.PRIORITIZATION_RESULTS_PATH
    )
)
(
    prioritization_results_df.writeTo(cfg.PRIORITIZATION_RESULTS_TABLE)
    .using("delta")
    .createOrReplace()
)

(
    prioritization_evaluation_df.write.format("delta").mode("overwrite").save(
        cfg.PRIORITIZATION_EVALUATION_PATH
    )
)
(
    prioritization_evaluation_df.writeTo(cfg.PRIORITIZATION_EVALUATION_TABLE)
    .using("delta")
    .createOrReplace()
)

print("Operational prioritization outputs saved successfully.")
print(f"Predictions table: {cfg.PREDICTIONS_TABLE}")
print(f"Prioritization results: {cfg.PRIORITIZATION_RESULTS_TABLE}")
print(f"Prioritization evaluation: {cfg.PRIORITIZATION_EVALUATION_TABLE}")


Operational prioritization outputs saved successfully.
Predictions table: workspace.default.flight_predictions
Prioritization results: workspace.default.flight_prioritization_results
Prioritization evaluation: workspace.default.flight_prioritization_evaluation
